In [21]:
from pathlib import Path
from ccdc.io import EntryReader
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
from ccdc import io

fda_drug_properties = Path(r"C:/Users/teaching/CCDC/Sandpit_CCDC2/Property_Calculations/fda_drug_properties.csv")
fda_drug_lipinski = Path(r"C:/Users/teaching/CCDC/Sandpit_CCDC2/Property_Calculations/fda_drug_lipinski.csv")

# Load the FDA approved drug subset from the CSD
drug_reader = io.EntryReader(subset=io.Subsets.DRUG)


def csd_iterator():

    #csd = EntryReader("csd")

    for entry in drug_reader:
        try:
            mol = entry.molecule
            smiles = mol.smiles
        except RuntimeError:
            continue
        yield mol.identifier, smiles


def _unused_filter(mol):
    mol_id = mol.identifier
    donors = [a for a in mol.atoms if a.is_donor]
    acceptors = [a for a in mol.atoms if a.is_acceptor]

    n_donors = len(donors)
    n_acceptors = len(acceptors)
    weight = mol.molecular_weight


# def lipinski (logp, mol_weight, hb_donors, hb_acceptors):
#     """Apply Lipinski's Rule of Five to filter records."""
#     return (
#         logp < 5 and
#         mol_weight < 500 and
#         hb_donors < 5 and
#         hb_acceptors < 10
#     )


def run():
    """Run the search and return a pandas DataFrame of computed records."""
    records = []
    for i, (identifier, smiles) in enumerate(csd_iterator()):
        if not smiles:
            continue
        mol = Chem.MolFromSmiles(smiles)

        if not mol:
            continue

        logp_crippen = Descriptors.MolLogP(mol)
        mol_weight = Descriptors.ExactMolWt(mol)
        hb_donors = Descriptors.NumHDonors(mol)
        hb_acceptors = Descriptors.NumHAcceptors(mol)

        record = {
            "identifier": identifier,
            "logp_crippen": logp_crippen,
            "mol_weight": mol_weight,
            "hb_donors": hb_donors,
            "hb_acceptors": hb_acceptors,
            "smiles": smiles,
        }
        records.append(record)

    df = pd.DataFrame(records)
    df.to_csv(fda_drug_properties, index=False)
    return df


def lipinski_filter(df):
    """Apply Lipinski's Rule of Five to filter records."""
    filtered_df = df[
        (df["logp_crippen"] < 5) &
        (df["mol_weight"] < 500) &
        (df["hb_donors"] < 5) &
        (df["hb_acceptors"] < 10)
    ].copy()

    filtered_df.to_csv(fda_drug_lipinski, index=False)
    return filtered_df


if __name__ == "__main__":
    df = run()
    lipinski_filter(df)

 

[15:34:56] WARNING: not removing hydrogen atom without neighbors
[15:34:56] WARNING: not removing hydrogen atom without neighbors
[15:34:56] Explicit valence for atom # 16 O, 4, is greater than permitted
[15:34:57] WARNING: not removing hydrogen atom without neighbors
[15:34:57] WARNING: not removing hydrogen atom without neighbors
[15:34:57] WARNING: not removing hydrogen atom without neighbors
[15:34:57] WARNING: not removing hydrogen atom without neighbors
[15:34:57] Explicit valence for atom # 48 C, 5, is greater than permitted
[15:34:58] WARNING: not removing hydrogen atom without neighbors
[15:34:58] WARNING: not removing hydrogen atom without neighbors
[15:34:58] WARNING: not removing hydrogen atom without neighbors
[15:34:58] WARNING: not removing hydrogen atom without neighbors
[15:34:58] WARNING: not removing hydrogen atom without neighbors
[15:34:58] Explicit valence for atom # 28 C, 5, is greater than permitted
[15:35:00] Explicit valence for atom # 6 C, 5, is greater than 

In [32]:
# variables are already loaded as DataFrames in the notebook; reuse them
df_lipinski = fda_drug_lipinski
df_properties = fda_drug_properties

# print(len(drug_reader))
# print(len(df_properties))
# print(len(df_lipinski))
print(f'Percentage of drug not parsed by RDKit: {(len(drug_reader) - len(df))/len(drug_reader) * 100:.2f}%')
print(f'Percentage of FDA approved drugs passing Lipinski filter: {len(df_lipinski)/len(df_properties) * 100:.2f}%')

# Calculate percentage of entries with <5 logP
# percentage_lt_5 = (df["logp_crippen"] < 5).mean() * 100

# print(f"Percentage with logP < 5: {percentage_lt_5:.2f}%")

Percentage of drug not parsed by RDKit: 3.02%
Percentage of FDA approved drugs passing Lipinski filter: 54.96%
